# MLflow Experiment Tracking with Model Registry — RHOAI Demo

**~15 minutes** | Train a model with MLflow experiment tracking, trace LLM calls, then register the model in Model Registry.

> This notebook demonstrates the full lifecycle: **train → track → store → register**.
> All artifacts and metadata flow through RHOAI-managed MLflow and Model Registry.

## Setup

In [ ]:
!pip install -q mlflow openai scikit-learn boto3 requests

In [ ]:
import os

# --- EDIT THIS: paste your MaaS API key ---
os.environ["MAAS_API_KEY"] = "<YOUR_MAAS_API_KEY>"

# Cluster endpoints
MAAS_URL = "https://maas.apps.ocp.xlwsd.sandbox1213.opentlc.com/rhoai-playground/qwen3-8b/v1"

# Use external route — the inline trace widget renders in the browser,
# which cannot resolve internal .svc addresses
MLFLOW_TRACKING_URI = "https://rh-ai.apps.ocp.xlwsd.sandbox1213.opentlc.com/mlflow/"

MODEL_REGISTRY_URL = "https://demo-registry.rhoai-model-registries.svc:8443"
MODEL_REGISTRY_API = f"{MODEL_REGISTRY_URL}/api/model_registry/v1alpha3"
MINIO_ENDPOINT = "http://minio-service.minio.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "pipelines"

# If running OUTSIDE the cluster, also uncomment this:
# MODEL_REGISTRY_URL = "https://demo-registry-rest.apps.ocp.xlwsd.sandbox1213.opentlc.com"
# MODEL_REGISTRY_API = f"{MODEL_REGISTRY_URL}/api/model_registry/v1alpha3"

# Auth token
try:
    with open("/var/run/secrets/kubernetes.io/serviceaccount/token") as f:
        TOKEN = f.read().strip()
    print("Using service account token")
except FileNotFoundError:
    import subprocess
    TOKEN = subprocess.check_output(["oc", "whoami", "-t"]).decode().strip()
    print("Using oc token")

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_TRACKING_TOKEN"] = TOKEN
print(f"MLflow: {MLFLOW_TRACKING_URI}")
print(f"Model Registry: {MODEL_REGISTRY_API}")

---
## Part 1: Train a sklearn Model with MLflow Tracking

Train a LogisticRegression on the Iris dataset. MLflow logs parameters, metrics, and the model artifact automatically.

In [ ]:
import mlflow
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_workspace("rhoai-playground")
mlflow.set_experiment("model-registry-demo")

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

C = 1.0
SOLVER = "lbfgs"
MAX_ITER = 200

with mlflow.start_run(run_name="iris-logistic-regression") as run:
    mlflow.log_params({"C": C, "solver": SOLVER, "max_iter": MAX_ITER, "dataset": "iris"})

    model = LogisticRegression(C=C, solver=SOLVER, max_iter=MAX_ITER, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    mlflow.log_metrics({"accuracy": accuracy, "f1_score": f1})
    mlflow.sklearn.log_model(model, artifact_path="model", input_example=X_test[:1])

    RUN_ID = run.info.run_id
    ARTIFACT_URI = run.info.artifact_uri

print(f"Run ID:       {RUN_ID}")
print(f"Artifact URI: {ARTIFACT_URI}")
print(f"Accuracy:     {accuracy:.4f}")
print(f"F1 Score:     {f1:.4f}")

---
## Part 2: LLM Evaluation with MLflow

Use `mlflow.openai.autolog()` to auto-trace LLM calls to qwen3-8b via MaaS. Log a quality metric.

In [ ]:
import mlflow
import openai
import time

mlflow.openai.autolog()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_workspace("rhoai-playground")
mlflow.set_experiment("model-registry-demo")

client = openai.OpenAI(
    base_url=MAAS_URL,
    api_key=os.environ["MAAS_API_KEY"],
)

with mlflow.start_run(run_name="llm-quality-check"):
    prompt = "Explain the difference between logistic regression and a neural network in 3 bullet points."
    response = client.chat.completions.create(
        model="qwen3-8b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.3,
    )

    output = response.choices[0].message.content
    print(f"LLM Response:\n{output}\n")

    # Manual quality score (1-5) — in production, use an LLM judge
    mlflow.log_metric("llm_response_quality", 4.0)
    mlflow.log_params({"model": "qwen3-8b", "temperature": 0.3, "task": "explain-ml-concepts"})

mlflow.flush_trace_async_logging(terminate=True)
time.sleep(3)
print("Traces logged. Check MLflow UI -> Traces tab.")

---
## Part 3: Save Model to MinIO

Persist the trained sklearn model to MinIO S3 storage. This creates a durable artifact URI for the Model Registry.

In [ ]:
import pickle
import io
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),
    region_name="us-east-1",
)

model_bytes = pickle.dumps(model)
model_key = "models/logistic_regression.pkl"

s3.put_object(
    Bucket=MINIO_BUCKET,
    Key=model_key,
    Body=io.BytesIO(model_bytes),
    ContentType="application/octet-stream",
)

MODEL_S3_URI = f"s3://{MINIO_BUCKET}/{model_key}"
print(f"Model saved to: {MODEL_S3_URI}")
print(f"Size: {len(model_bytes)} bytes")

---
## Part 4: Register in Model Registry

Push the model metadata to the RHOAI Model Registry via REST API. Three steps:
1. Register the model
2. Create a version
3. Attach the artifact (S3 URI)

In [ ]:
import requests
import json
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}

# Step 1: Register the model
reg_model = requests.post(
    f"{MODEL_REGISTRY_API}/registered_models",
    headers=headers,
    json={
        "name": "mlflow-logistic-regression",
        "description": "Iris LogisticRegression trained with MLflow experiment tracking",
        "customProperties": {
            "framework": {"stringValue": "scikit-learn"},
            "task": {"stringValue": "classification"},
            "source": {"stringValue": "mlflow-experiment"},
        },
    },
    verify=False,
)
reg_model.raise_for_status()
model_id = reg_model.json()["id"]
print(f"Registered model ID: {model_id}")

# Step 2: Create a version
version = requests.post(
    f"{MODEL_REGISTRY_API}/registered_models/{model_id}/versions",
    headers=headers,
    json={
        "name": "v1",
        "description": f"MLflow run {RUN_ID} — accuracy={accuracy:.4f}",
        "customProperties": {
            "mlflow_run_id": {"stringValue": RUN_ID},
            "accuracy": {"doubleValue": accuracy},
            "f1_score": {"doubleValue": f1},
        },
    },
    verify=False,
)
version.raise_for_status()
version_id = version.json()["id"]
print(f"Model version ID:   {version_id}")

# Step 3: Attach the artifact
artifact = requests.post(
    f"{MODEL_REGISTRY_API}/model_versions/{version_id}/artifacts",
    headers=headers,
    json={
        "name": "logistic-regression-pkl",
        "uri": MODEL_S3_URI,
        "description": "Pickled sklearn LogisticRegression model",
        "modelFormatName": "sklearn",
        "modelFormatVersion": "1.0",
        "customProperties": {
            "mlflow_experiment": {"stringValue": "model-registry-demo"},
            "mlflow_run_id": {"stringValue": RUN_ID},
            "storage_backend": {"stringValue": "minio"},
        },
    },
    verify=False,
)
artifact.raise_for_status()
artifact_id = artifact.json()["id"]
print(f"Artifact ID:        {artifact_id}")
print(f"\nModel registered with S3 URI: {MODEL_S3_URI}")

---
## Verify in Model Registry

In [ ]:
import requests
import json

# Fetch registered models
resp = requests.get(
    f"{MODEL_REGISTRY_API}/registered_models",
    headers={"Authorization": f"Bearer {TOKEN}"},
    verify=False,
)
resp.raise_for_status()
models = resp.json()

print("=== Registered Models ===")
for m in models.get("items", []):
    print(f"  [{m['id']}] {m['name']} — {m.get('description', '')}")

# Fetch versions for our model
resp = requests.get(
    f"{MODEL_REGISTRY_API}/registered_models/{model_id}/versions",
    headers={"Authorization": f"Bearer {TOKEN}"},
    verify=False,
)
resp.raise_for_status()
versions = resp.json()

print(f"\n=== Versions for model {model_id} ===")
for v in versions.get("items", []):
    print(f"  [{v['id']}] {v['name']} — {v.get('description', '')}")

# Fetch artifacts for our version
resp = requests.get(
    f"{MODEL_REGISTRY_API}/model_versions/{version_id}/artifacts",
    headers={"Authorization": f"Bearer {TOKEN}"},
    verify=False,
)
resp.raise_for_status()
artifacts = resp.json()

print(f"\n=== Artifacts for version {version_id} ===")
for a in artifacts.get("items", []):
    print(f"  [{a['id']}] {a.get('name', 'unnamed')} -> {a.get('uri', 'no uri')}")

---
## What to Show in the UI

### MLflow Dashboard
1. Open **MLflow** → Develop & train → Experiments in the RHOAI dashboard
2. Select workspace **rhoai-playground** (top-left dropdown)
3. Click experiment **model-registry-demo**
4. Two runs: `iris-logistic-regression` (training) and `llm-quality-check` (LLM eval)
5. Click `iris-logistic-regression` → see logged parameters (C, solver), metrics (accuracy, f1_score), and model artifact
6. Check the **Traces** tab for auto-captured LLM call traces

### Model Registry Dashboard
1. Open **Model Registry** in the RHOAI dashboard
2. Select registry **demo-registry**
3. Find **mlflow-logistic-regression** → version **v1**
4. Check the artifact URI points to `s3://pipelines/models/logistic_regression.pkl`
5. Custom properties show the MLflow run ID — linking the registry entry back to the experiment

> **Key message**: MLflow tracks the *experiment* (how the model was trained, what metrics it achieved).
> Model Registry tracks the *artifact* (where the model lives, which version is deployed).
> Together, they give you full lineage from training to deployment.